In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/soil_types/plsda/soil_types.csv', sep=';') # local copy of Toledo and Guarapuava soil datasets
data = data_complete.loc[:, '1.32':'13.1']

# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1.32':'13.1'], test_size=0.30) # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1.32':'13.1'], test_size=0.30) # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True) # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0]) # creating the target variable for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0]) # creating the target variable for prediction set

# preprocessings
import preprocessings as prepr # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-08 19:34:32,544 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
2026-02-08 19:34:32,623 - kennard_stone.utils._pairwise:109[INFO] - Calculating pairwise distances using scikit-learn.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\deprecation.py:151: Futur

In [2]:
from modeling import svm_optimized

svm_model = svm_optimized(Xcalclass_prep, ycalclass, Xpredclass_prep, ypredclass, aim='classification', kernel='rbf')
svm_model[0]

,Model,Accuracy Cal,Sensitivity Cal,Specificity Cal,CM Cal,Accuracy Pred,Sensitivity Pred,Specificity Pred,CM Pred
0,SVC,1.0,1.0,1.0,"[[247, 0], [0, 254]]",1.0,1.0,1.0,"[[107, 0], [0, 109]]"


## Definição das Zonas Espectrais

In [3]:
spectral_cuts = [
('Al', 1.33, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.19),
('S', 2.19, 2.55),
('Rh L + Ar', 2.55, 3.21),
('K', 3.21, 3.53),
('Ca ka', 3.53, 3.84),
('Ca kb', 3.84, 4.37),
('Ti ka', 4.37, 4.75),
('Ti kb', 4.75, 5.12),
('Cr', 5.12, 5.77),
('Mn', 5.77, 6.13),
('Fe ka', 6.13, 6.80),
('Fe kb', 6.80, 7.30),
('Ni', 7.30, 7.91),
('Cu', 7.91, 8.20),
('Zn', 8.20, 9.0),
('background1', 9.0, 10.69),
('Fe ka + Ti ka', 10.69, 11.14),
('background2', 11.14, 12.55),
('sum Fe' , 12.55, 13.1),
]

## Pvector and SHAP

In [4]:
X_sv = svm_model[3].support_vectors_            # shape (n_SV, n_features)
alpha_dual = svm_model[3].dual_coef_.ravel()    # shape (n_SV,)

# Calculando os coeficientes p usando os vetores de suporte e os multiplicadores de Lagrange
pvetor = pd.DataFrame({'energia' : Xcalclass.columns,
                       'importance': (X_sv.T) @ alpha_dual})
pvetor['importance'] = np.abs(pvetor['importance'])
pvector_df = pd.DataFrame({
    'energy': pvetor['energia'],
    'Pvector': pvetor['importance'].values
})
pvector_df = pvector_df.sort_values(by='Pvector', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in pvector_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
pvector_df['Zone'] = pvector_df['energy'].map(energy_to_zone_vip)
pvector_unique_df = pvector_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
pvector_unique_df

,energy,Pvector,Zone
0,6.42,38.192949,Fe ka
1,7.1,13.648747,Fe kb
2,3.32,7.823500,K
3,5.92,4.147603,Mn
4,12.84,3.515694,sum Fe
5,2.62,3.346369,Rh L + Ar
6,1.74,3.190167,Si
7,4.98,3.110417,Ti kb
8,8.02,2.888905,Cu
9,4.48,2.277795,Ti ka


In [5]:
shap_global_importance = pd.read_csv('shap_soil_type.csv', sep=';') # loading previously saved shap_unique_df
# vamos gerar uma nova coluna em shap_global_importance com o nome da zona espectral correspondente de acordo com a lista spectral_cuts
energy_to_zone_shap = {}
for zone_name, start, end in spectral_cuts:
    for i in shap_global_importance['energy']:
        i_float = float(i)
        if start <= i_float <= end:
            energy_to_zone_shap[i] = zone_name
shap_global_importance['Zone'] = shap_global_importance['energy'].map(energy_to_zone_shap)

# agora vamos filtrar shap_global_importance para manter apenas as zonas espectrais únicas com maior SHAP score
shap_unique_df = shap_global_importance.sort_values(by='Mean_Abs_SHAP', ascending=False).reset_index(drop=True)
shap_unique_df = shap_unique_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
shap_unique_df

,energy,Mean_Abs_SHAP,Zone
0,6.42,0.066660,Fe ka
1,4.50,0.001514,Ti ka
2,3.70,0.000280,Ca ka
3,7.08,0.000226,Fe kb
4,4.80,0.000213,Ti kb
5,7.60,0.000203,Ni
6,3.04,0.000201,Rh L + Ar
7,2.40,0.000157,S
8,10.22,0.000138,background1
9,8.10,0.000137,Cu


## Funções de Agregação por PCA

Aqui implementamos as funções para agregar zonas espectrais usando PCA com 1 componente principal.

In [6]:
import explaining as exp

# Extração das zonas espectrais
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_scores_df, pca_info_dict = exp.aggregate_spectral_zones_pca(spectral_zones_class) # apca aggregation
print(f"\nScores DataFrame shape: {zone_scores_df.shape}")

Zona 'Al': VE = 87.97%, dim = 15 variáveis
Zona 'Si': VE = 96.43%, dim = 12 variáveis
Zona 'P': VE = 62.58%, dim = 17 variáveis
Zona 'S': VE = 44.25%, dim = 18 variáveis
Zona 'Rh L + Ar': VE = 73.64%, dim = 33 variáveis
Zona 'K': VE = 95.35%, dim = 16 variáveis
Zona 'Ca ka': VE = 98.88%, dim = 16 variáveis
Zona 'Ca kb': VE = 74.22%, dim = 27 variáveis
Zona 'Ti ka': VE = 85.88%, dim = 19 variáveis
Zona 'Ti kb': VE = 88.37%, dim = 19 variáveis
Zona 'Cr': VE = 82.96%, dim = 33 variáveis
Zona 'Mn': VE = 94.39%, dim = 18 variáveis
Zona 'Fe ka': VE = 91.33%, dim = 34 variáveis
Zona 'Fe kb': VE = 90.72%, dim = 26 variáveis
Zona 'Ni': VE = 58.25%, dim = 31 variáveis
Zona 'Cu': VE = 84.29%, dim = 15 variáveis
Zona 'Zn': VE = 60.20%, dim = 41 variáveis
Zona 'background1': VE = 61.89%, dim = 85 variáveis
Zona 'Fe ka + Ti ka': VE = 55.37%, dim = 23 variáveis
Zona 'background2': VE = 50.79%, dim = 71 variáveis
Zona 'sum Fe': VE = 90.84%, dim = 28 variáveis

Scores DataFrame shape: (501, 21)


In [7]:
y_pred = svm_model[4]['SVC'].values # using the continuous predictions from SVM, extracting as 1D array
predicates_quantiles = exp.predicates_by_quantiles(zone_scores_df, [0.2, 0.4, 0.6, 0.8])
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_scores_df,
    y_predicted_numeric=y_pred
)

In [8]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

y_pred = pd.Series(svm_model[4]['SVC'].values) # using the continuous predictions from SVM, extracting as 1D array

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_cov[seed]['bags_result'],
        predicate_ranking_dict=all_results_cov[seed]['cov_results_dict'],
        metric_column='Covariance',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | 

,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Fe ka > -9.83,Fe ka > -9.83,Fe ka > -9.83,Fe ka > -9.83
1,Fe ka <= 11.65,Fe ka <= 11.65,Fe ka <= 11.65,Fe ka <= 11.65
2,Fe ka <= 4.25,Fe ka > -5.77,Fe ka > -5.77,Fe ka > -5.77
3,Fe ka > -5.77,Fe ka <= 4.25,Fe ka <= 4.25,Fe ka <= 4.25
4,Fe kb > -3.67,Fe kb > -3.67,Fe kb > -3.67,Fe kb > -3.67
...,...,...,...,...
105,NaN,Zn <= -0.08,Class_A,NaN
106,NaN,Ni <= -0.06,Class_B,NaN
107,NaN,S <= -0.07,NaN,NaN
108,NaN,Class_A,NaN,NaN


In [9]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_cov_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_cov = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_cov = lrc_summed_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_cov = lrc_summed_df_cov.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
#lrc_summed_unique_df_cov = lrc_summed_unique_df_cov.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_cov

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > -9.83,20.420307,Fe ka,-9.83,>
1,Fe kb > -3.67,8.444839,Fe kb,-3.67,>
2,Mn > -1.21,3.888434,Mn,-1.21,>
3,sum Fe > -1.11,3.184910,sum Fe,-1.11,>
4,Ti ka > -1.99,0.992262,Ti ka,-1.99,>
5,Rh L + Ar <= 0.76,0.980888,Rh L + Ar,0.76,<=
6,Ti kb > -1.01,0.928389,Ti kb,-1.01,>
7,Ca ka > -1.89,0.883664,Ca ka,-1.89,>
8,Si <= 1.00,0.856164,Si,1.00,<=
9,Cr <= 0.70,0.764329,Cr,0.70,<=


In [10]:
# Criar zone_sums_df para dados NÃO pré-processados (Xcalclass)
spectral_zones_original = exp.extract_spectral_zones(Xcalclass, spectral_cuts)
zones_original = exp.aggregate_spectral_zones_pca(spectral_zones_original) # apca aggregation

# Aplicar o mapeamento
lrc_summed_df_cov_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_cov,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_cov_natural

Zona 'Al': VE = 94.39%, dim = 15 variáveis
Zona 'Si': VE = 97.95%, dim = 12 variáveis
Zona 'P': VE = 63.93%, dim = 17 variáveis
Zona 'S': VE = 50.04%, dim = 18 variáveis
Zona 'Rh L + Ar': VE = 82.78%, dim = 33 variáveis
Zona 'K': VE = 97.13%, dim = 16 variáveis
Zona 'Ca ka': VE = 99.41%, dim = 16 variáveis
Zona 'Ca kb': VE = 73.60%, dim = 27 variáveis
Zona 'Ti ka': VE = 92.52%, dim = 19 variáveis
Zona 'Ti kb': VE = 91.98%, dim = 19 variáveis
Zona 'Cr': VE = 83.92%, dim = 33 variáveis
Zona 'Mn': VE = 95.74%, dim = 18 variáveis
Zona 'Fe ka': VE = 94.87%, dim = 34 variáveis
Zona 'Fe kb': VE = 94.17%, dim = 26 variáveis
Zona 'Ni': VE = 58.62%, dim = 31 variáveis
Zona 'Cu': VE = 85.47%, dim = 15 variáveis
Zona 'Zn': VE = 60.21%, dim = 41 variáveis
Zona 'background1': VE = 61.93%, dim = 85 variáveis
Zona 'Fe ka + Ti ka': VE = 58.87%, dim = 23 variáveis
Zona 'background2': VE = 51.14%, dim = 71 variáveis
Zona 'sum Fe': VE = 94.31%, dim = 28 variáveis


,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Fe ka > -9.83,20.420307,Fe ka,-9.83,>,-178.952975,247.0,0.001731,Fe ka > -178.952975
1,Fe ka <= 11.65,15.678693,Fe ka,11.65,<=,188.786502,94.0,0.001158,Fe ka <= 188.786502
2,Fe ka > -5.77,10.387961,Fe ka,-5.77,>,-103.046579,408.0,0.004763,Fe ka > -103.046579
3,Fe ka <= 4.25,9.511881,Fe ka,4.25,<=,87.045387,38.0,0.001079,Fe ka <= 87.045387
4,Fe kb > -3.67,8.444839,Fe kb,-3.67,>,-25.194168,311.0,0.001842,Fe kb > -25.194168
...,...,...,...,...,...,...,...,...,...
106,S > 0.04,0.000232,S,0.04,>,0.036958,195.0,0.000109,S > 0.036958
107,Ni <= -0.06,0.000217,Ni,-0.06,<=,-0.094370,282.0,0.000178,Ni <= -0.094370
108,Zn <= -0.08,0.000151,Zn,-0.08,<=,-0.077538,384.0,0.000025,Zn <= -0.077538
109,Class_A,0.000000,None,None,None,NaN,NaN,NaN,None


## Reconstrução de Thresholds Multivariados

Agora vamos pegar os thresholds dos predicados mais importantes e reconstruí-los como espectros completos.

In [11]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_cov_natural.iloc[n]['Zone']
threshold_score = float(lrc_summed_df_cov_natural.iloc[n]['Threshold_Natural'])  # Corrigido: usa n em vez de 7

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
print(f"  - Range de energias: {threshold_spectrum.index[0]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_cov_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Fe ka':
  - Dimensão: 34 variáveis espectrais
  - Range de energias: 6.14 - 6.8
  - Variância explicada pela PC1: 94.87%


In [12]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_scores_df,
        y_predicted_numeric=y_pred,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=svm_model[3],
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        #perturbation_value=0,
        perturbation_mode='median', # valores entre 'mean' ou 'min'
        stats_source='full', # full indica usar todas as amostras para calcular estatísticas enquanto que 'fold' usa apenas as amostras do fold atual
        #metric='mean_abs_diff',   # Média com sinal (pode ser negativo)
        aim='classification',
        metric='probability_shift', 
        verbose=True
    )

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    
    # Construir grafo para esta semente (sem predicates_df - extraído da regra)
    DG = exp.build_predicate_graphv3_pca(
        bags_result=all_results_pert[seed]['bags_result'],
        predicate_ranking_dict=all_results_pert[seed]['pert_results_dict'],
        metric_column='Perturbation',
        random_state=seed,
        show_details=True,
        var_exp=True,
        pca_info_dict=pca_info_dict
    )

    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 126 | Descartados: 42
PERTURBATION IMPORTANCE PARA PREDICADOS
Tipo de tarefa (aim): classification
Modo de perturbação: median
Fonte das estatísticas: full
Métrica: probability_shift
Tot

,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Fe ka > 4.25,Fe ka > 4.25,Fe ka > 4.25,Fe ka > 4.25
1,Fe ka > -5.77,Fe ka > -5.77,Fe ka > -5.77,Fe ka > -5.77
2,Fe ka <= -5.77,Fe ka <= -5.77,Fe ka <= -5.77,Fe ka <= -5.77
3,Fe ka > -9.83,Fe ka > -9.83,Fe ka > -9.83,Fe ka > -9.83
4,Fe ka <= 11.65,Fe ka <= 11.65,Fe ka <= 11.65,Fe ka <= 11.65
...,...,...,...,...
123,S > 0.04,S <= -0.07,S > -0.16,S <= -0.07
124,S <= 0.04,K <= -0.19,K <= -0.19,S > -0.16
125,S <= -0.07,K <= -0.01,S <= -0.07,S <= 0.04
126,Class_A,Class_A,Class_A,Class_A


In [13]:
all_results_pert[0]['pert_results_dict']['Bag_1']

,Predicate,Perturbation
0,Fe ka > 4.25,0.453564
1,Fe ka > -5.77,0.327683
2,Fe ka <= -5.77,0.288999
3,Fe ka > -9.83,0.271674
4,Fe ka <= 11.65,0.251057
...,...,...
121,S <= 0.04,0.000021
122,S <= -0.07,0.000018
123,K <= 0.22,0.000017
124,K <= -0.01,0.000013


In [14]:
# Somar LRCs de predicados equivalentes entre diferentes seeds
lrc_combined_list = []

for seed in random_seeds:
    lrc_df = lrc_pert_by_seed[seed].copy()
    lrc_combined_list.append(lrc_df) # o append adiciona o dataframe ao final da lista

# Concatenar todos os dataframes
lrc_all_seeds = pd.concat(lrc_combined_list, ignore_index=True) 

# Agrupar por predicado (Node) e somar as LRCs, mantendo Zone, Threshold e Operator
lrc_summed_df_pert = lrc_all_seeds.groupby('Node').agg({ # o .agg pode ser usado para aplicar múltiplas funções de agregação
    'Local_Reaching_Centrality': 'mean', # sum = somando as LRCs, poderia ser média ou outro agregado
    'Zone': 'first',
    'Threshold': 'first',
    'Operator': 'first'
}).reset_index()

# Ordenar pelo valor de LRC somado (maior para menor)
lrc_summed_df_pert = lrc_summed_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
# vamoss pegar so os valore sunicos de lrc_summed_df baseado na zona espectral
lrc_summed_unique_df_pert = lrc_summed_df_pert.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
lrc_summed_unique_df_pert = lrc_summed_unique_df_pert.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
lrc_summed_unique_df_pert

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator
0,Fe ka > 4.25,33.726066,Fe ka,4.25,>
1,Ti ka <= -0.55,0.512573,Ti ka,-0.55,<=
2,Fe kb <= -2.08,0.356791,Fe kb,-2.08,<=
3,Ca ka > -0.35,0.318930,Ca ka,-0.35,>
4,K > -0.01,0.229622,K,-0.01,>
5,Ca kb <= 0.45,0.139571,Ca kb,0.45,<=
6,Si > 0.21,0.120229,Si,0.21,>
7,P <= 0.30,0.090481,P,0.30,<=
8,Cu <= -0.21,0.056745,Cu,-0.21,<=
9,Rh L + Ar <= 0.33,0.054529,Rh L + Ar,0.33,<=


In [15]:
# Aplicar o mapeamento
lrc_summed_df_pert_natural = exp.map_thresholds_to_natural(
    lrc_df=lrc_summed_df_pert,
    zone_sums_preprocessed=zone_scores_df,
    zone_sums_natural=zones_original[0]
)

lrc_summed_df_pert_natural

,Node,Local_Reaching_Centrality,Zone,Threshold,Operator,Threshold_Natural,Reference_Sample_Index,Approximation_Error,Node_Natural
0,Fe ka > 4.25,33.726066,Fe ka,4.25,>,87.045387,38.0,0.001079,Fe ka > 87.045387
1,Fe ka > -5.77,25.653554,Fe ka,-5.77,>,-103.046579,408.0,0.004763,Fe ka > -103.046579
2,Fe ka <= -5.77,20.269064,Fe ka,-5.77,<=,-103.046579,408.0,0.004763,Fe ka <= -103.046579
3,Fe ka > -9.83,15.032997,Fe ka,-9.83,>,-178.952975,247.0,0.001731,Fe ka > -178.952975
4,Fe ka <= 11.65,9.493358,Fe ka,11.65,<=,188.786502,94.0,0.001158,Fe ka <= 188.786502
...,...,...,...,...,...,...,...,...,...
123,K <= -0.19,0.000066,K,-0.19,<=,-0.266533,231.0,0.000657,K <= -0.266533
124,S <= 0.04,0.000061,S,0.04,<=,0.036958,195.0,0.000109,S <= 0.036958
125,S <= -0.07,0.000055,S,-0.07,<=,-0.047908,438.0,0.001616,S <= -0.047908
126,Class_B,0.000000,None,None,None,NaN,NaN,NaN,None


In [16]:
import plotly.graph_objects as go

n = 0
zone_name = lrc_summed_df_pert_natural.iloc[n]['Zone'] # o n indica a linha do dataframe lrc_summed_df_pert_natural que queremos acessar para pegar o nome da zona espectral (coluna 'Zone')
threshold_score = float(lrc_summed_df_pert_natural.iloc[n]['Threshold_Natural']) # o iloc acessa a linha n e a coluna 'Threshold_Natural' para pegar o valor do threshold já mapeado para o espaço natural (não pré-processado)

# Usar pca_info_dict dos dados ORIGINAIS (zones_original[1])
pca_info_dict_original = zones_original[1]

# Reconstrução: τ = mean + threshold * loadings (no espaço original)
threshold_spectrum = exp.reconstruct_threshold_to_spectrum(
    threshold_value=threshold_score,
    zone_name=zone_name,
    pca_info_dict=pca_info_dict_original
)
print(f"\nEspectro de threshold reconstruído para zona '{zone_name}':")
print(f"  - Dimensão: {len(threshold_spectrum)} variáveis espectrais")
#print(f"  - Range de energias: {threshold_spectrum.index[n]} - {threshold_spectrum.index[-1]}")
print(f"  - Variância explicada pela PC1: {pca_info_dict_original[zone_name]['variance_explained']:.2%}")

# Usar dados ORIGINAIS para plotar (spectral_zones_original)
zone_df = spectral_zones_original[zone_name]

# Plotar zona com threshold (inline)
fig = go.Figure()
x_values = pd.to_numeric(zone_df.columns, errors='coerce')

# Plotar espectros das amostras coloridas por classe
colors = {'A': 'gold', 'B': 'blue'}
for idx, row in zone_df.iterrows():
    class_label = ycalclass.iloc[idx] if idx < len(ycalclass) else 'Unknown'
    fig.add_trace(go.Scatter(
        x=x_values,
        y=row.values,
        mode='lines',
        line=dict(color=colors.get(class_label, 'rgba(128,128,128,0.3)'), width=0.5),
        name=f'Class {class_label}',
        showlegend=False,
        hoverinfo='skip'
    ))

# Plotar espectro de threshold (destaque)
fig.add_trace(go.Scatter(
    x=x_values,
    y=threshold_spectrum.values,
    mode='lines',
    line=dict(color='red', width=4, dash='dash'),
    name=f'Threshold Spectrum ({threshold_spectrum.name})'
))

# Layout do gráfico
fig.update_layout(
    title=f"Zona '{zone_name}' com Threshold Multivariado (Predicado: {lrc_summed_df_pert_natural.iloc[n]['Node_Natural']})",
    xaxis_title='Energia / Comprimento de Onda',
    yaxis_title='Intensidade',
    template='plotly_white',
    showlegend=True,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()


Espectro de threshold reconstruído para zona 'Fe ka':
  - Dimensão: 34 variáveis espectrais
  - Variância explicada pela PC1: 94.87%


In [17]:
# Permutation importance baseado em mudança nas probabilidades previstas (predict_proba)
# Medimos a média da diferença absoluta entre as probabilidades originais e as probabilidades
# obtidas após permutar cada variável. Isso fornece uma métrica contínua de importância.
n_repeats = 10
rng = np.random.RandomState(42)
# Probabilidades base (classe 'B' mapeada para 1)
baseline_proba = svm_model[3].predict_proba(Xcalclass_prep)[:, 1]
importance_list = []
X_arr = Xcalclass_prep.copy()
for col in Xcalclass_prep.columns:
    diffs = []
    for _ in range(n_repeats):
        X_perm = X_arr.copy()
        X_perm[col] = rng.permutation(X_perm[col].values)
        perm_proba = svm_model[3].predict_proba(X_perm)[:, 1]
        diffs.append(np.mean(np.abs(baseline_proba - perm_proba)))
    importance_list.append(np.mean(diffs))

permutation_df = pd.DataFrame({
    'energy': Xcalclass_prep.columns,
    'Permutation_importance_proba': importance_list
})
permutation_df.sort_values(by='Permutation_importance_proba', ascending=False, inplace=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in permutation_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
permutation_df['Zone'] = permutation_df['energy'].map(energy_to_zone_vip)
permutation_unique_df = permutation_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
permutation_unique_df = permutation_unique_df.sort_values(by='Permutation_importance_proba', ascending=False)
permutation_unique_df

,energy,Permutation_importance_proba,Zone
0,6.42,0.012566,Fe ka
1,4.52,0.000823,Ti ka
2,7.1,0.000752,Fe kb
3,3.68,0.000578,Ca ka
4,3.32,0.000181,K
5,5.92,0.000173,Mn
6,1.72,0.000108,Si
7,1.48,0.000108,Al
8,2.62,0.000063,Rh L + Ar
9,8.02,0.000058,Cu


In [18]:
import numpy as np

max_len = max(
    len(pvector_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(permutation_unique_df['Zone']),
    len(lrc_summed_unique_df_pert['Zone']),
    len(lrc_summed_unique_df_cov['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'SVM_pvector': pad_list(pvector_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'Permutation' : pad_list(permutation_unique_df['Zone'], max_len),
    'LRC_perturbation' : pad_list(lrc_summed_unique_df_pert['Zone'], max_len),
    'LRC_covariance' : pad_list(lrc_summed_unique_df_cov['Zone'], max_len),
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,SVM_pvector,Shap,Permutation,LRC_perturbation,LRC_covariance
0,Fe ka,Fe ka,Fe ka,Fe ka,Fe ka
1,Fe kb,Ti ka,Ti ka,Ti ka,Fe kb
2,K,Ca ka,Fe kb,Fe kb,Mn
3,Mn,Fe kb,Ca ka,Ca ka,sum Fe
4,sum Fe,Ti kb,K,K,Ti ka
5,Rh L + Ar,Ni,Mn,Ca kb,Rh L + Ar
6,Si,Rh L + Ar,Si,Si,Ti kb
7,Ti kb,S,Al,P,Ca ka
8,Cu,background1,Rh L + Ar,Cu,Si
9,Ti ka,Cu,Cu,Rh L + Ar,Cr


In [19]:
import rbo
rbo_comparison = pd.DataFrame(columns=['Method_1', 'Method_2', 'RBO_Score'])
methods = features_importance.columns.tolist()
for i in range(len(methods)):
    for j in range(i + 1, len(methods)):
        method_1 = methods[i]
        method_2 = methods[j]
        # Remove None values from the lists
        list_1 = [x for x in features_importance[method_1].tolist() if x is not None]
        list_2 = [x for x in features_importance[method_2].tolist() if x is not None]
        # Truncate both lists to the same length (minimum of both)
        min_len = min(len(list_1), len(list_2))
        list_1_trunc = list_1[:min_len]
        list_2_trunc = list_2[:min_len]
        rbo_score = rbo.RankingSimilarity(list_1_trunc, list_2_trunc).rbo(p=0.7, k=10)
        rbo_comparison = pd.concat([rbo_comparison, pd.DataFrame({
            'Method_1': [method_1],
            'Method_2': [method_2],
            'RBO_Score': [rbo_score]
        })], ignore_index=True)
rbo_comparison.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_comparison.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_comparison

C:\Users\Usuario\AppData\Local\Temp\ipykernel_18316\3097109587.py:16: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



,Method_1,Method_2,RBO_Score
7,Permutation,LRC_perturbation,0.943944
3,SVM_pvector,LRC_covariance,0.851693
4,Shap,Permutation,0.851531
5,Shap,LRC_perturbation,0.849610
1,SVM_pvector,Permutation,0.693148
8,Permutation,LRC_covariance,0.688817
2,SVM_pvector,LRC_perturbation,0.673482
9,LRC_perturbation,LRC_covariance,0.667230
6,Shap,LRC_covariance,0.636412
0,SVM_pvector,Shap,0.593498


In [20]:
lrc_summed_df_cov_natural.to_csv('lrc_cov_natural.csv', index=False, sep=';')
lrc_summed_df_pert_natural.to_csv('lrc_pert_natural.csv', index=False, sep=';')